In [ ]:
# ============================================================
# Frozen DINO-adapted SigLIP2 for binary polyp segmentation
#
# Input: segmented-images/images/<image>
# Ground truth: segmented-images/masks/<same filename>
#
# Model:
#   frozen DINO-adapted SigLIP2 encoder
#   + trainable lightweight segmentation decoder
#
# Task:
#   pixel-wise binary classification
#       0 = background
#       1 = polyp
#
# Loss:
#   BCEWithLogitsLoss + Dice Loss
#
# Validation:
#   Dice, IoU, Precision, Recall, Pixel Accuracy
#
# Important:
#   - bounding-boxes.json is NOT used.
#   - encoder is completely frozen.
#   - only the segmentation decoder is trained.
#   - segmentation checkpoint saves decoder weights only.
# ============================================================

from pathlib import Path
import random
import gc

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as TF
from torchvision.transforms import InterpolationMode

from sklearn.model_selection import train_test_split
from transformers import AutoProcessor, AutoModel


# ============================================================
# 0. Config
# ============================================================

DATA_ROOT = Path("/kaggle/input/datasets/kelkalot/the-hyper-kvasir-dataset")
SEG_ROOT = DATA_ROOT / "segmented-images"
IMAGE_DIR = SEG_ROOT / "images"
MASK_DIR = SEG_ROOT / "masks"

OUTPUT_DIR = Path("/kaggle/working/frozen_dino_siglip2_polyp_segmentation")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "google/siglip2-base-patch16-224"

# CHANGE THIS PATH.
DINO_CKPT_PATH = Path("/kaggle/input/notebooks/lilyii70/dinov2-style-siglip/siglip2_dino_style_unlabeled_full/siglip2_dino_style_epoch3.pt")
DINO_STATE_KEY = "student_state_dict"

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

RANDOM_STATE = 42
TRAIN_RATIO = 0.80

IMAGE_SIZE = 224
PATCH_SIZE = 16

BATCH_SIZE = 16
NUM_WORKERS = 2

NUM_EPOCHS = 30
LR = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 7

BCE_WEIGHT = 1.0
DICE_WEIGHT = 1.0

MASK_THRESHOLD = 0.5

DECODER_CHANNELS = 256
DROPOUT = 0.10

USE_AUGMENTATION = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Image directory:", IMAGE_DIR)
print("Mask directory:", MASK_DIR)
print("DINO checkpoint:", DINO_CKPT_PATH)


# ============================================================
# 1. Seed
# ============================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_STATE)


# ============================================================
# 2. Build image / mask index
# ============================================================

def build_segmentation_index():
    if not IMAGE_DIR.exists():
        raise FileNotFoundError(f"Image directory not found: {IMAGE_DIR}")

    if not MASK_DIR.exists():
        raise FileNotFoundError(f"Mask directory not found: {MASK_DIR}")

    image_by_stem = {p.stem: p for p in IMAGE_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS}
    mask_by_stem = {p.stem: p for p in MASK_DIR.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTS}

    common_stems = sorted(set(image_by_stem) & set(mask_by_stem))

    if len(common_stems) == 0:
        raise ValueError("No matching image/mask pairs found.")

    rows = []
    for stem in common_stems:
        rows.append({
            "image_id": stem,
            "image_path": str(image_by_stem[stem]),
            "mask_path": str(mask_by_stem[stem]),
        })

    df = pd.DataFrame(rows)

    missing_masks = sorted(set(image_by_stem) - set(mask_by_stem))
    missing_images = sorted(set(mask_by_stem) - set(image_by_stem))

    print("\n========== Segmentation dataset ==========")
    print("Images:", len(image_by_stem))
    print("Masks:", len(mask_by_stem))
    print("Matched image/mask pairs:", len(df))
    print("Images without masks:", len(missing_masks))
    print("Masks without images:", len(missing_images))

    df.to_csv(OUTPUT_DIR / "all_segmentation_pairs.csv", index=False)

    return df


# ============================================================
# 3. Train / validation split
# ============================================================

def make_train_val_split(df):
    train_df, val_df = train_test_split(
        df,
        train_size=TRAIN_RATIO,
        random_state=RANDOM_STATE,
        shuffle=True,
    )

    train_df = train_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    train_df.to_csv(OUTPUT_DIR / "train_df.csv", index=False)
    val_df.to_csv(OUTPUT_DIR / "val_df.csv", index=False)

    print("\n========== Train / validation split ==========")
    print("Train:", len(train_df))
    print("Validation:", len(val_df))

    return train_df, val_df


# ============================================================
# 4. Synchronized image / mask transforms
# ============================================================

def apply_train_transform(image, mask):
    image = TF.resize(image, [IMAGE_SIZE, IMAGE_SIZE], interpolation=InterpolationMode.BICUBIC)
    mask = TF.resize(mask, [IMAGE_SIZE, IMAGE_SIZE], interpolation=InterpolationMode.NEAREST)

    if random.random() < 0.5:
        image = TF.hflip(image)
        mask = TF.hflip(mask)

    if random.random() < 0.15:
        image = TF.vflip(image)
        mask = TF.vflip(mask)

    if random.random() < 0.30:
        angle = random.uniform(-8.0, 8.0)
        image = TF.rotate(image, angle=angle, interpolation=InterpolationMode.BILINEAR, fill=0)
        mask = TF.rotate(mask, angle=angle, interpolation=InterpolationMode.NEAREST, fill=0)

    if random.random() < 0.30:
        brightness = random.uniform(0.90, 1.10)
        contrast = random.uniform(0.90, 1.10)
        saturation = random.uniform(0.90, 1.10)

        image = TF.adjust_brightness(image, brightness)
        image = TF.adjust_contrast(image, contrast)
        image = TF.adjust_saturation(image, saturation)

    return image, mask


def apply_eval_transform(image, mask):
    image = TF.resize(image, [IMAGE_SIZE, IMAGE_SIZE], interpolation=InterpolationMode.BICUBIC)
    mask = TF.resize(mask, [IMAGE_SIZE, IMAGE_SIZE], interpolation=InterpolationMode.NEAREST)
    return image, mask


# ============================================================
# 5. Dataset
# ============================================================

class PolypSegmentationDataset(Dataset):
    def __init__(self, df, processor, training=False):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.training = training

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        with Image.open(row["image_path"]) as img:
            image = img.convert("RGB")

        with Image.open(row["mask_path"]) as m:
            mask = m.convert("L")

        if self.training and USE_AUGMENTATION:
            image, mask = apply_train_transform(image, mask)
        else:
            image, mask = apply_eval_transform(image, mask)

        processed = self.processor(images=image, return_tensors="pt")
        pixel_values = processed["pixel_values"].squeeze(0)

        mask_array = np.asarray(mask, dtype=np.float32)
        mask_array = (mask_array >= 127.5).astype(np.float32)
        mask_tensor = torch.from_numpy(mask_array).unsqueeze(0)

        return {
            "pixel_values": pixel_values,
            "mask": mask_tensor,
            "image_id": str(row["image_id"]),
            "image_path": str(row["image_path"]),
            "mask_path": str(row["mask_path"]),
        }


# ============================================================
# 6. Decoder
# ============================================================

class ConvNormAct(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.GELU(),
        )

    def forward(self, x):
        return self.block(x)


class SegmentationDecoder(nn.Module):
    def __init__(self, hidden_dim, decoder_channels=256, dropout=0.10):
        super().__init__()

        self.projection = nn.Sequential(
            nn.Conv2d(hidden_dim, decoder_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(decoder_channels),
            nn.GELU(),
        )

        self.up1 = ConvNormAct(decoder_channels, 256)
        self.up2 = ConvNormAct(256, 128)
        self.up3 = ConvNormAct(128, 64)
        self.up4 = ConvNormAct(64, 32)

        self.dropout = nn.Dropout2d(dropout)
        self.head = nn.Conv2d(32, 1, kernel_size=1)

    def forward(self, feature_map, output_size=(224, 224)):
        x = self.projection(feature_map)

        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = self.up1(x)

        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = self.up2(x)

        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = self.up3(x)

        x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
        x = self.up4(x)

        x = self.dropout(x)
        logits = self.head(x)

        if logits.shape[-2:] != output_size:
            logits = F.interpolate(logits, size=output_size, mode="bilinear", align_corners=False)

        return logits

def show_segmentation_result(image_path, mask):
    image = np.array(Image.open(image_path).convert("RGB"))

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(image)

    ax.contour(
        mask,
        levels=[0.5],
        linewidths=2,
    )

    ax.axis("off")
    plt.show()

    return fig
# ============================================================
# 7. Frozen DINO-SigLIP2 segmentation model
# ============================================================

class FrozenDinoSigLIP2SegmentationModel(nn.Module):
    def __init__(
        self,
        model_name=MODEL_NAME,
        dino_checkpoint_path=DINO_CKPT_PATH,
        state_key=DINO_STATE_KEY,
        decoder_channels=DECODER_CHANNELS,
        dropout=DROPOUT,
    ):
        super().__init__()

        if not Path(dino_checkpoint_path).exists():
            raise FileNotFoundError(f"DINO checkpoint not found: {dino_checkpoint_path}")

        print("\n========== Load DINO-adapted SigLIP2 encoder ==========")

        self.encoder = AutoModel.from_pretrained(
            model_name,
            torch_dtype=torch.float32,
            trust_remote_code=True,
        )

        checkpoint = torch.load(
            dino_checkpoint_path,
            map_location="cpu",
            weights_only=False,
        )

        if state_key not in checkpoint:
            raise KeyError(
                f"DINO checkpoint does not contain '{state_key}'. "
                f"Available keys: {list(checkpoint.keys())}"
            )

        missing, unexpected = self.encoder.load_state_dict(checkpoint[state_key], strict=False)

        print("Loaded DINO checkpoint:", dino_checkpoint_path)
        print("DINO checkpoint epoch:", checkpoint.get("epoch"))
        print("DINO SSL train loss:", checkpoint.get("ssl_train_loss"))
        print("Missing encoder keys:", len(missing))
        print("Unexpected encoder keys:", len(unexpected))

        for parameter in self.encoder.parameters():
            parameter.requires_grad = False

        self.encoder.eval()

        if hasattr(self.encoder.config, "vision_config"):
            hidden_dim = int(self.encoder.config.vision_config.hidden_size)
        elif hasattr(self.encoder.vision_model.config, "hidden_size"):
            hidden_dim = int(self.encoder.vision_model.config.hidden_size)
        else:
            raise RuntimeError("Could not determine SigLIP2 vision hidden dimension.")

        self.hidden_dim = hidden_dim
        self.decoder = SegmentationDecoder(
            hidden_dim=hidden_dim,
            decoder_channels=decoder_channels,
            dropout=dropout,
        )

        del checkpoint
        gc.collect()

    def train(self, mode=True):
        super().train(mode)

        # Encoder must stay frozen and in eval mode.
        self.encoder.eval()

        return self

    def tokens_to_feature_map(self, tokens):
        batch_size, token_count, hidden_dim = tokens.shape

        expected_grid = IMAGE_SIZE // PATCH_SIZE
        expected_tokens = expected_grid * expected_grid

        if token_count == expected_tokens:
            patch_tokens = tokens
            grid_size = expected_grid

        elif token_count == expected_tokens + 1:
            patch_tokens = tokens[:, 1:, :]
            grid_size = expected_grid

        else:
            grid_size = int(round(token_count ** 0.5))

            if grid_size * grid_size != token_count:
                raise RuntimeError(
                    f"Cannot reshape {token_count} vision tokens into a square feature map."
                )

            patch_tokens = tokens

        feature_map = patch_tokens.transpose(1, 2).reshape(
            batch_size,
            hidden_dim,
            grid_size,
            grid_size,
        )

        return feature_map

    def forward(self, pixel_values):
        with torch.no_grad():
            vision_outputs = self.encoder.vision_model(
                pixel_values=pixel_values,
                output_hidden_states=True,
                return_dict=True,
            )

            tokens = vision_outputs.last_hidden_state

        feature_map = self.tokens_to_feature_map(tokens)

        logits = self.decoder(
            feature_map,
            output_size=(IMAGE_SIZE, IMAGE_SIZE),
        )

        return logits


# ============================================================
# 8. Dice loss
# ============================================================

def dice_loss_from_logits(logits, targets, smooth=1.0):
    probabilities = torch.sigmoid(logits)

    probabilities = probabilities.flatten(start_dim=1)
    targets = targets.flatten(start_dim=1)

    intersection = (probabilities * targets).sum(dim=1)

    dice_score = (
        2.0 * intersection + smooth
    ) / (
        probabilities.sum(dim=1) + targets.sum(dim=1) + smooth
    )

    return (1.0 - dice_score).mean()


# ============================================================
# 9. BCE + Dice loss
# ============================================================

def segmentation_loss(logits, targets):
    bce = F.binary_cross_entropy_with_logits(logits, targets)
    dice = dice_loss_from_logits(logits, targets)

    total = BCE_WEIGHT * bce + DICE_WEIGHT * dice

    return total, bce, dice


# ============================================================
# 10. Metrics
# ============================================================

@torch.no_grad()
def compute_batch_metrics(logits, targets, threshold=0.5, eps=1e-7):
    probabilities = torch.sigmoid(logits)
    predictions = (probabilities >= threshold).float()

    predictions = predictions.flatten(start_dim=1)
    targets = targets.flatten(start_dim=1)

    tp = (predictions * targets).sum(dim=1)
    fp = (predictions * (1.0 - targets)).sum(dim=1)
    fn = ((1.0 - predictions) * targets).sum(dim=1)
    tn = ((1.0 - predictions) * (1.0 - targets)).sum(dim=1)

    dice = (2.0 * tp + eps) / (2.0 * tp + fp + fn + eps)
    iou = (tp + eps) / (tp + fp + fn + eps)
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    pixel_accuracy = (tp + tn + eps) / (tp + tn + fp + fn + eps)

    return {
        "dice": dice.cpu().numpy(),
        "iou": iou.cpu().numpy(),
        "precision": precision.cpu().numpy(),
        "recall": recall.cpu().numpy(),
        "pixel_accuracy": pixel_accuracy.cpu().numpy(),
    }


# ============================================================
# 11. Validation
# ============================================================

@torch.no_grad()
def evaluate_model(model, loader):
    model.eval()

    losses = []
    bce_losses = []
    dice_losses = []

    dice_values = []
    iou_values = []
    precision_values = []
    recall_values = []
    pixel_accuracy_values = []

    prediction_rows = []

    for batch in tqdm(loader, desc="Validation", leave=False):
        pixel_values = batch["pixel_values"].to(DEVICE, non_blocking=True)
        targets = batch["mask"].to(DEVICE, non_blocking=True)

        logits = model(pixel_values)

        loss, bce, dice_loss = segmentation_loss(logits, targets)
        metrics = compute_batch_metrics(logits, targets, threshold=MASK_THRESHOLD)

        losses.append(float(loss.item()))
        bce_losses.append(float(bce.item()))
        dice_losses.append(float(dice_loss.item()))

        dice_values.extend(metrics["dice"].tolist())
        iou_values.extend(metrics["iou"].tolist())
        precision_values.extend(metrics["precision"].tolist())
        recall_values.extend(metrics["recall"].tolist())
        pixel_accuracy_values.extend(metrics["pixel_accuracy"].tolist())

        for i in range(len(batch["image_id"])):
            prediction_rows.append({
                "image_id": batch["image_id"][i],
                "image_path": batch["image_path"][i],
                "mask_path": batch["mask_path"][i],
                "dice": float(metrics["dice"][i]),
                "iou": float(metrics["iou"][i]),
                "precision": float(metrics["precision"][i]),
                "recall": float(metrics["recall"][i]),
                "pixel_accuracy": float(metrics["pixel_accuracy"][i]),
            })

    results = {
        "loss": float(np.mean(losses)),
        "bce_loss": float(np.mean(bce_losses)),
        "dice_loss": float(np.mean(dice_losses)),
        "dice": float(np.mean(dice_values)),
        "iou": float(np.mean(iou_values)),
        "precision": float(np.mean(precision_values)),
        "recall": float(np.mean(recall_values)),
        "pixel_accuracy": float(np.mean(pixel_accuracy_values)),
    }

    prediction_df = pd.DataFrame(prediction_rows)

    return results, prediction_df


# ============================================================
# 12. Save predicted masks
# ============================================================

@torch.no_grad()
def save_validation_predictions(model, dataset, epoch, max_images=20):
    out_dir = OUTPUT_DIR / "validation_predictions" / f"epoch_{epoch:03d}"
    out_dir.mkdir(parents=True, exist_ok=True)

    model.eval()

    count = min(max_images, len(dataset))

    for idx in range(count):
        sample = dataset[idx]

        pixel_values = sample["pixel_values"].unsqueeze(0).to(DEVICE)
        logits = model(pixel_values)
        probability = torch.sigmoid(logits)[0, 0].cpu().numpy()
        prediction = (probability >= MASK_THRESHOLD).astype(np.uint8)

        Image.fromarray(prediction * 255).save(
            out_dir / f"{sample['image_id']}_pred.png"
        )


# ============================================================
# 13. Training
# ============================================================

def train_segmentation_model(model, train_loader, val_loader, val_dataset):
    encoder_trainable = sum(
        p.numel()
        for p in model.encoder.parameters()
        if p.requires_grad
    )

    decoder_trainable = sum(
        p.numel()
        for p in model.decoder.parameters()
        if p.requires_grad
    )

    print("\n========== Trainable parameters ==========")
    print("Encoder trainable:", encoder_trainable)
    print("Decoder trainable:", f"{decoder_trainable:,}")

    if encoder_trainable != 0:
        raise RuntimeError("Encoder is supposed to be completely frozen.")

    optimizer = torch.optim.AdamW(
        model.decoder.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    best_dice = -1.0
    best_epoch = -1
    patience = 0
    history = []

    best_checkpoint_path = OUTPUT_DIR / "best_segmentation_head.pt"

    print("\n========== Start segmentation training ==========")

    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()

        train_losses = []
        train_bce_losses = []
        train_dice_losses = []

        pbar = tqdm(
            train_loader,
            desc=f"Train epoch {epoch}/{NUM_EPOCHS}",
        )

        for batch in pbar:
            pixel_values = batch["pixel_values"].to(DEVICE, non_blocking=True)
            targets = batch["mask"].to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            logits = model(pixel_values)
            loss, bce, dice = segmentation_loss(logits, targets)

            if not torch.isfinite(loss):
                raise RuntimeError("Segmentation loss became NaN/Inf.")

            loss.backward()
            optimizer.step()

            train_losses.append(float(loss.item()))
            train_bce_losses.append(float(bce.item()))
            train_dice_losses.append(float(dice.item()))

            pbar.set_postfix(
                loss=f"{np.mean(train_losses):.4f}",
                bce=f"{np.mean(train_bce_losses):.4f}",
                dice=f"{np.mean(train_dice_losses):.4f}",
            )

        val_metrics, val_prediction_df = evaluate_model(model, val_loader)

        row = {
            "epoch": epoch,
            "train_loss": float(np.mean(train_losses)),
            "train_bce_loss": float(np.mean(train_bce_losses)),
            "train_dice_loss": float(np.mean(train_dice_losses)),
            "val_loss": val_metrics["loss"],
            "val_bce_loss": val_metrics["bce_loss"],
            "val_dice_loss": val_metrics["dice_loss"],
            "val_dice": val_metrics["dice"],
            "val_iou": val_metrics["iou"],
            "val_precision": val_metrics["precision"],
            "val_recall": val_metrics["recall"],
            "val_pixel_accuracy": val_metrics["pixel_accuracy"],
        }

        history.append(row)

        pd.DataFrame(history).to_csv(
            OUTPUT_DIR / "training_history.csv",
            index=False,
        )

        val_prediction_df.to_csv(
            OUTPUT_DIR / f"val_predictions_epoch_{epoch:03d}.csv",
            index=False,
        )

        print(
            f"\nEpoch {epoch:03d} | "
            f"train_loss={row['train_loss']:.4f} | "
            f"val_loss={row['val_loss']:.4f} | "
            f"Dice={row['val_dice']:.4f} | "
            f"IoU={row['val_iou']:.4f} | "
            f"Precision={row['val_precision']:.4f} | "
            f"Recall={row['val_recall']:.4f} | "
            f"PixelAcc={row['val_pixel_accuracy']:.4f}"
        )

        save_validation_predictions(
            model=model,
            dataset=val_dataset,
            epoch=epoch,
            max_images=20,
        )

        if row["val_dice"] > best_dice:
            best_dice = row["val_dice"]
            best_epoch = epoch
            patience = 0

            torch.save(
                {
                    "model_name": MODEL_NAME,
                    "dino_checkpoint_path": str(DINO_CKPT_PATH),
                    "dino_state_key": DINO_STATE_KEY,
                    "encoder_frozen": True,
                    "image_size": IMAGE_SIZE,
                    "patch_size": PATCH_SIZE,
                    "decoder_channels": DECODER_CHANNELS,
                    "dropout": DROPOUT,
                    "mask_threshold": MASK_THRESHOLD,
                    "best_epoch": best_epoch,
                    "best_val_dice": best_dice,
                    "decoder_state_dict": {
                        k: v.detach().cpu().clone()
                        for k, v in model.decoder.state_dict().items()
                    },
                    "history": history.copy(),
                },
                best_checkpoint_path,
            )

            print("Updated best segmentation head:", best_checkpoint_path)

        else:
            patience += 1

        if patience >= PATIENCE:
            print(
                f"\nEarly stopping | "
                f"best_epoch={best_epoch} | "
                f"best_val_dice={best_dice:.4f}"
            )
            break

    print("\n========== Segmentation training complete ==========")
    print("Best epoch:", best_epoch)
    print("Best validation Dice:", best_dice)
    print("Best segmentation head:", best_checkpoint_path)

    return {
        "history": pd.DataFrame(history),
        "best_epoch": best_epoch,
        "best_val_dice": best_dice,
        "best_checkpoint_path": best_checkpoint_path,
    }


# ============================================================
# 14. Load trained segmentation model
# ============================================================

def load_trained_segmentation_model(
    segmentation_checkpoint,
    dino_checkpoint=None,
    device=None,
):
    device = torch.device(
        device or (
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )
    )

    segmentation_ckpt = torch.load(
        segmentation_checkpoint,
        map_location="cpu",
        weights_only=False,
    )

    if dino_checkpoint is None:
        dino_checkpoint = segmentation_ckpt["dino_checkpoint_path"]

    model = FrozenDinoSigLIP2SegmentationModel(
        model_name=segmentation_ckpt["model_name"],
        dino_checkpoint_path=dino_checkpoint,
        state_key=segmentation_ckpt["dino_state_key"],
        decoder_channels=segmentation_ckpt["decoder_channels"],
        dropout=segmentation_ckpt["dropout"],
    )

    model.decoder.load_state_dict(
        segmentation_ckpt["decoder_state_dict"],
        strict=True,
    )

    model.to(device).eval()

    return model, segmentation_ckpt


# ============================================================
# 15. Single-image inference
# ============================================================

@torch.no_grad()
def predict_segmentation(
    model,
    processor,
    image,
    threshold=MASK_THRESHOLD,
    device=DEVICE,
):
    if isinstance(image, (str, Path)):
        with Image.open(image) as img:
            image = img.convert("RGB")

    elif isinstance(image, Image.Image):
        image = image.convert("RGB")

    else:
        raise TypeError(
            "image must be a path or PIL.Image.Image"
        )

    original_height = image.height
    original_width = image.width

    resized = TF.resize(
        image,
        [IMAGE_SIZE, IMAGE_SIZE],
        interpolation=InterpolationMode.BICUBIC,
    )

    processed = processor(
        images=resized,
        return_tensors="pt",
    )

    pixel_values = processed["pixel_values"].to(device)

    logits = model(pixel_values)

    probability = torch.sigmoid(logits)

    probability = F.interpolate(
        probability,
        size=(original_height, original_width),
        mode="bilinear",
        align_corners=False,
    )

    probability = probability[0, 0].cpu().numpy()

    mask = (
        probability >= threshold
    ).astype(np.uint8)

    return {
        "probability": probability,
        "mask": mask,
    }

@torch.no_grad()
def predict_and_show_segmentation(
    model,
    processor,
    image_path,
    threshold=MASK_THRESHOLD,
    device=DEVICE,
):
    result = predict_segmentation(
        model=model,
        processor=processor,
        image=image_path,
        threshold=threshold,
        device=device,
    )

    fig = show_segmentation_result(
        image_path=image_path,
        mask=result["mask"],
    )

    return {
        "probability": result["probability"],
        "mask": result["mask"],
        "figure": fig,
    }

# ============================================================
# 16. Main pipeline
# ============================================================

def run_segmentation_pipeline():
    set_seed(RANDOM_STATE)

    all_df = build_segmentation_index()
    train_df, val_df = make_train_val_split(all_df)

    print("\n========== Load SigLIP2 processor ==========")

    processor = AutoProcessor.from_pretrained(
        MODEL_NAME,
        trust_remote_code=True,
    )

    train_dataset = PolypSegmentationDataset(
        train_df,
        processor=processor,
        training=True,
    )

    val_dataset = PolypSegmentationDataset(
        val_df,
        processor=processor,
        training=False,
    )

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,
        pin_memory=torch.cuda.is_available(),
    )

    print("\n========== Build frozen DINO-SigLIP2 segmentation model ==========")

    model = FrozenDinoSigLIP2SegmentationModel(
        model_name=MODEL_NAME,
        dino_checkpoint_path=DINO_CKPT_PATH,
        state_key=DINO_STATE_KEY,
        decoder_channels=DECODER_CHANNELS,
        dropout=DROPOUT,
    ).to(DEVICE)

    results = train_segmentation_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        val_dataset=val_dataset,
    )

    return {
        "model": model,
        "processor": processor,
        "train_df": train_df,
        "val_df": val_df,
        **results,
    }


# ============================================================
# 17. Run
# ============================================================

segmentation_results = run_segmentation_pipeline()